In [ ]:
import sys
from pathlib import Path
root = Path.cwd()
if not (root / 'kis').is_dir():
    root = root.parent
sys.path.insert(0, str(root))
from kis.web_app import SearchService, find_keyframe_image
import os
from getpass import getpass
if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass('Groq API key: ').strip()
search = SearchService()
search.status()

In [ ]:
raw_query = 'Hai người dẫn chương trình đứng sau bàn trong trường quay'
response = search.search(raw_query)
print(response['video_groups'], response['fusion_method'], response['elapsed_seconds'])

In [ ]:
import pandas as pd
results = response['results']
pd.DataFrame(results)[['rank', 'video_id', 'frame_id', 'image_number', 'score', 'caption', 'vlm_status']]

In [ ]:
from IPython.display import Image, display
for item in results:
    print(item['rank'], item['video_id'], item['frame_id'], item['caption'])
    if item['image_number'] is not None:
        if search.image_source is not None:
            data, _ = search.image_source.read(item['video_id'], item['image_number'])
            display(Image(data=data, width=480))
            del data
        else:
            image = find_keyframe_image(item['video_id'], item['image_number'])
            if image is not None:
                display(Image(filename=str(image), width=480))